# MCP, agents, etc.

## AGENTS


The field of AI agents has shifted rapidly from "chatbots with tools" to Autonomous Systems and Multi-Agent Orchestration. In 2026, the theory focuses less on how a single model thinks and more on how systems of models collaborate, remember, and self-correct.

The most significant shift in 2026 is the move from "Single Agent" to "Agent Swarms."

Theory: Instead of one large model trying to do everything, tasks are decomposed into specialized nodes.

Emergent Intelligence: Like a colony of ants, simple individual agents follow local rules to solve complex global problems.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
open_api_key = os.getenv("OPEN_API_KEY")

os.environ['OPENAI_API_KEY'] = open_api_key

# 1. SETUP
client = OpenAI(api_key=open_api_key)

# 2. THE TOOL: A simple python function
def calculate(expression):
    """Calculates a mathematical expression."""
    try:
        # Warning: eval() is used here for educational simplicity; 
        # In real apps, use a math library for safety!
        return str(eval(expression))
    except:
        return "Error: Could not calculate."

# 3. THE AGENT LOGIC
def run_math_agent(question):
    # System prompt tells the AI how to use the 'tool'
    system_prompt = (
        "You are a math agent. If you need to calculate something, "
        "respond ONLY with: CALC:[expression]. I will give you the result, "
        "and then you can provide the final answer."
    )

    # Step 1: The Agent "Thinks"
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )
    
    ai_message = response.choices[0].message.content

    # Step 2: The Agent "Acts" (If it sees the CALC: keyword)
    if "CALC:" in ai_message:
        expression = ai_message.split("CALC:")[1].strip()
        print(f"--- Agent is using a tool for: {expression} ---")
        
        result = calculate(expression)
        
        # Step 3: Final response with the tool result
        final_response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": question},
                {"role": "assistant", "content": ai_message},
                {"role": "user", "content": f"Tool Result: {result}"}
            ]
        )
        return final_response.choices[0].message.content
    
    return ai_message

# EXECUTION
if __name__ == "__main__":
    user_query = "What is 1234 multiplied by 5678, and then add 100?"
    print(f"User: {user_query}")
    print(f"Agent: {run_math_agent(user_query)}")

User: What is 1234 multiplied by 5678, and then add 100?
--- Agent is using a tool for: (1234 * 5678) + 100 ---
Agent: The result of 1234 multiplied by 5678, and then adding 100, is 7,006,752.


Orchestration: New theories on "Conductor" agents that manage the workflow, passing "state" and "context" between worker agents to prevent "hallucination loops."

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
open_api_key = os.getenv("OPEN_API_KEY")

os.environ['OPENAI_API_KEY'] = open_api_key

client = OpenAI(api_key=open_api_key)

# --- AGENT 1: The Specialist Researcher ---
def researcher_agent(topic):
    print("🔎 [Researcher]: Looking up facts about:", topic)
    # In a real app, this would be a web search tool call.
    facts = {
        "AI Agents": "Agents use LLMs to reason and call tools. 2026 is the year of Agent Swarms.",
        "Python": "Python 3.13+ has major performance improvements for AI workloads."
    }
    return facts.get(topic, "General fact: Agents are cool.")

# --- AGENT 2: The Specialist Writer ---
def writer_agent(facts):
    print("✍️ [Writer]: Writing a story based on facts...")
    prompt = f"Turn these facts into a catchy 2-sentence marketing blurb: {facts}"
    response = client.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

# --- THE ORCHESTRATOR (The Supervisor) ---
def run_orchestration(user_request):
    print(f"🚀 [Supervisor]: New task received: {user_request}")
    
    # Logic Step 1: Decide to delegate to Researcher
    # (In complex systems, the LLM makes this decision. Here we show the flow.)
    topic = "AI Agents" 
    research_results = researcher_agent(topic)
    
    # Logic Step 2: Pass Researcher's output to the Writer
    final_copy = writer_agent(research_results)
    
    # Logic Step 3: Final Quality Check
    print("✅ [Supervisor]: Content verified. Delivering to user.")
    return final_copy

# EXECUTION
if __name__ == "__main__":
    result = run_orchestration("Write a blurb about AI Agents")
    print(f"\nFINAL OUTPUT:\n{result}")

🚀 [Supervisor]: New task received: Write a blurb about AI Agents
🔎 [Researcher]: Looking up facts about: AI Agents
✍️ [Writer]: Writing a story based on facts...
✅ [Supervisor]: Content verified. Delivering to user.

FINAL OUTPUT:
Unlock the future with Agent Swarms, where sophisticated LLMs come together to seamlessly reason and harness tools for you in 2026. Embrace innovation and stay ahead with the power of collective intelligence!


## MCP

MCP (Model Context Protocol) is an open standard introduced by Anthropic in November 2024 for connecting AI assistants to data systems such as content repositories, business tools, and development environments. Before MCP, developers had to build custom connectors for each data source or tool, resulting in what Anthropic described as an "N×M" data integration problem.

MCP servers expose data through Resources (information retrieval that returns data but doesn't execute computations), Tools (actions that can perform side effects like calculations or API requests), and Prompts (reusable templates for LLM-server communication). The transport layer uses JSON-RPC 2.0 for two-way message conversion.

In [3]:
# !pip install fastmcp
from fastmcp import FastMCP
import platform
import psutil # For system stats
import datetime

# 1. Initialize the MCP Server
# This creates the 'bridge' between the LLM and your notebook environment
mcp = FastMCP("PresentationDemo")

# 2. Add a 'Tool' (Action)
# Tools are functions the AI can choose to run
@mcp.tool()
def get_system_specs() -> str:
    """Returns the hardware specs and current OS of the presenter's machine."""
    uname = platform.uname()
    return f"OS: {uname.system}, Node: {uname.node}, CPU: {uname.processor}"

# 3. Add a 'Resource' (Static Data)
# Resources are like 'read-only' files the AI can look at
@mcp.resource("system://time")
def get_current_time() -> str:
    """Provides the current server time."""
    return f"The current system time is: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

print("✅ MCP Server 'PresentationDemo' is defined and ready to be hosted!")

✅ MCP Server 'PresentationDemo' is defined and ready to be hosted!


In [4]:
# SIMULATED INTERACTION (How an AI uses your server)
print("--- AI Discovery Mode ---")
# The AI 'scans' the server and finds the tools you just wrote
tools = await mcp.list_tools()
for t in tools:
    print(f"Found Tool: {t.name} - Description: {t.description}")

print("\n--- AI Execution Mode ---")
# If a user asks: 'What computer is this?', the AI calls the tool:
result = mcp.call_tool("get_system_specs", {})
print(f"AI received from system: {result}")

--- AI Discovery Mode ---
Found Tool: get_system_specs - Description: Returns the hardware specs and current OS of the presenter's machine.

--- AI Execution Mode ---
AI received from system: <coroutine object FastMCP.call_tool at 0x000001684F0EEB60>


Think of MCP as a USB-C port for AI systems — just as USB-C standardizes how devices connect to computers, MCP standardizes how AI agents access external resources like databases, APIs, file systems, and knowledge bases.

An MCP Server is the "other end of that cable." It's a small program that wraps a data source or capability — a database, a file system, a weather API, a calendar — and exposes it in a standard language that any LLM can understand and talk to. The LLM doesn't need to know how the database works, just that the MCP server speaks the protocol.

Before MCP, developers had to build custom connectors for each data source or tool, resulting in what Anthropic described as an "N×M" data integration problem. MCP re-uses the message-flow ideas of the Language Server Protocol and is transported over JSON-RPC 2.0.